In [1]:
from Montreal_UHI_toolbox import full_obs, full_obs_urban, full_obs_suburban, full_obs_rural
import plotly.graph_objects as go

/runoff/gulley/.miniconda3/lib/python3.12/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.39.0 or higher is recommended. You are running version 2.14.1
  warnings.warn(


In [ ]:
# for obs_set,obs_type in zip([obs, obs_urban, obs_rural, obs_suburban],['All','Urban','Rural','Suburban']):
for obs_set,obs_type in zip([full_obs_urban, full_obs_rural, full_obs_suburban],['Urban','Rural','Suburban']):
    for season in ['JJA','SON','DJF','MAM']:

        # Loading observation data 
        tasmax = obs_set.sel(time=slice('1980','2025'))['tasmax'].groupby('time.season')[season].mean('station').groupby('time.year').mean('time')
        tasmin = obs_set.sel(time=slice('1980','2025'))['tasmin'].groupby('time.season')[season].mean('station').groupby('time.year').mean('time')
        tas = obs_set.sel(time=slice('1980','2025'))['tas'].groupby('time.season')[season].mean('station').groupby('time.year').mean('time')

        tasmin_clim = tasmin.sel(year=slice('1980','2010')).mean('year')
        tasmax_clim = tasmax.sel(year=slice('1980','2010')).mean('year')
        tas_clim = tas.sel(year=slice('1980','2010')).mean('year')

        tasmin_anom = tasmin - tasmin_clim
        tasmax_anom = tasmax - tasmax_clim
        tas_anom = tas - tas_clim

        # for T, T_anom, T_clim, name,y_symbol in zip([tasmin,tas,tasmax],
        #     [tasmin_anom,tas_anom,tasmax_anom],
        #     [tasmin_clim,tas_clim,tasmax_clim],
        #     ['Daily Minimum','Daily Average','Daily Maximum'],
        #     ['T<sub>min</sub>','T<sub>avg</sub>','T<sub>max</sub>']):

        for T, T_anom, T_clim, name,y_symbol in zip([tasmin,tasmax],
            [tasmin_anom,tasmax_anom],
            [tasmin_clim,tasmax_clim],
            ['Daily Minimum','Daily Maximum'],
            ['<SPAN STYLE="text-decoration:overline">T</SPAN><sub>min</sub>','<SPAN STYLE="text-decoration:overline">T</SPAN><sub>max</sub>']):
            

            years = T.year.values

            hover_text = [
                f"Time: {y} ({season})<br>Temperature: {t:.2f}°C<br>Anomaly: {a:+.2f}°C"
                for y, t, a in zip(years, T, T_anom)
            ]

            fig = go.Figure(
                data=[
                    go.Bar(
                        name=name,
                        x=years,
                        y=T_anom,
                        marker_color='black',
                        hovertext=hover_text,
                        hoverinfo='text'
                    )
                ]
            )

            fig.add_vrect(
                x0=1979.6, x1=2010.4,                  
                fillcolor='lightgrey', opacity=0.35,
                line_width=0
            )

            fig.add_annotation(
                x=(1980+2010)/2 - 1,              
                y=0.95,                
                text=f'1980-2010 Climatology ({T_clim.values:.2f}°C)',  
                showarrow=False,
                arrowhead=2,
                ax=0,               
                ay=40,
                yref='paper'
            )
            
            fig.add_annotation(
                x=(2011+2023)/2,              
                y=0.95,                
                text=f'2011-2023 Average Anomaly ({T.sel(year=slice('2011','2023')).mean('year')
        .values - T_clim.values:+.2f} °C)',  
                showarrow=False,
                arrowhead=2,
                ax=0,               
                ay=40,
                yref='paper')
            
            fig.update_layout(title= dict(text=f'{season} {y_symbol} Anomaly for {obs_type} Montréal Stations',
                                        xanchor='center',
                                        x=0.5),
                            xaxis_title='Year', 
                            yaxis_title=f'Anomaly {y_symbol} (°C)',
                            barmode='group',
                            yaxis_range=[-5,5])

            # fig.show()
            fig.write_html(f'/runoff/gulley/UHI_plots/climatology/obs_climatology_{season}_{obs_type}_{T.name}.html')